# NOtas

## Variables derivadas: fabricar una columna nueva con una fórmula

In [1]:
import pandas as pd
df = pd.read_csv("../../data/S12_PD_Diaz_DatasetIntegrado.csv")

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 570976 entries, 0 to 570975
Data columns (total 66 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id_pedido                    570976 non-null  str    
 1   id_cliente                   570976 non-null  str    
 2   fecha_pedido                 550688 non-null  str    
 3   ciudad_tienda                570976 non-null  str    
 4   codigo_postal                570976 non-null  int64  
 5   producto                     570976 non-null  str    
 6   precio_unitario              570976 non-null  float64
 7   unidades_vendidas            570976 non-null  float64
 8   monto_compra                 570976 non-null  float64
 9   edad_cliente                 570976 non-null  float64
 10  correo_cliente               477145 non-null  str    
 11  nivel_satisfaccion           570976 non-null  str    
 12  nivel_lealtad                570976 non-null  str    
 13  comentario

## features dañadas 

el tutochimbo nos dice que agreguemos estas features , pero usa variables que no estan en el data set para hacerlo entonces agragamos features para poder hacer features 

#### feature dañada 1. dias desde ultima compra sin fecha ultima compra 

In [3]:
#Día última compra
#import pandas as pd

#hoy = pd.Timestamp("2026-10-19")  # fecha de referencia del corte de análisis
#df["dias_desde_ultima_compra"] = (hoy - df["fecha_ultima_compra"]).dt.days


In [4]:
hoy = pd.Timestamp("2026-10-19")

df["fecha_pedido"] = pd.to_datetime(df["fecha_pedido"], errors="coerce")

print("fecha_pedido nulas:", df["fecha_pedido"].isna().sum())
print("rango:", df["fecha_pedido"].min(), "->", df["fecha_pedido"].max())
print("pedidos posteriores al corte:", (df["fecha_pedido"] > hoy).sum())
print("id_cliente nulos:", df["id_cliente"].isna().sum(), "| clientes únicos:", df["id_cliente"].nunique())

fecha_pedido nulas: 20288
rango: 2025-01-01 00:00:00 -> 2027-12-28 00:00:00
pedidos posteriores al corte: 14694
id_cliente nulos: 0 | clientes únicos: 151117


In [5]:
manana = pd.Timestamp("2026-11-02")

print("posteriores al corte pero <= hoy:", ((df["fecha_pedido"] > hoy) & (df["fecha_pedido"] <= manana)).sum())
print("posteriores a hoy (imposibles):  ", (df["fecha_pedido"] > manana).sum())
print(df["fecha_pedido"].dt.year.value_counts().sort_index())

# Clientes que quedarían sin fecha_ultima_compra (todos sus pedidos nulos o posteriores al corte)
validas = df.loc[df["fecha_pedido"] <= hoy]
sin_recencia = set(df["id_cliente"]) - set(validas["id_cliente"])
print("clientes sin ninguna fecha válida <= corte:", len(sin_recencia))

posteriores al corte pero <= hoy:

 0
posteriores a hoy (imposibles):   14694
fecha_pedido
2025.0    392127
2026.0    143867
2027.0     14694
Name: count, dtype: int64
clientes sin ninguna fecha válida <= corte: 993


In [6]:
ultima = (
    df.loc[df["fecha_pedido"] <= hoy]
      .groupby("id_cliente")["fecha_pedido"].max()
      .rename("fecha_ultima_compra")
)

n0 = len(df)
df = df.merge(ultima, on="id_cliente", how="left", validate="m:1")
assert len(df) == n0

df["dias_desde_ultima_compra"] = (hoy - df["fecha_ultima_compra"]).dt.days
print(df["dias_desde_ultima_compra"].describe())

count    569878.000000
mean        262.496436
std          95.574646
min         157.000000
25%         189.000000
50%         234.000000
75%         308.000000
max         656.000000
Name: dias_desde_ultima_compra, dtype: float64


In [7]:
df["fecha_pedido_futura"] = df["fecha_pedido"] > pd.Timestamp("2026-11-02")
print("última fecha válida:", df.loc[~df["fecha_pedido_futura"], "fecha_pedido"].max())

ultima = (
    df.loc[df["fecha_pedido"] <= hoy]
      .groupby("id_cliente")["fecha_pedido"].max()
)

df["fecha_ultima_compra"] = df["id_cliente"].map(ultima)

df["dias_desde_ultima_compra"] = (hoy - df["fecha_ultima_compra"]).dt.days
df["dias_desde_ultima_compra_era_nulo"] = df["dias_desde_ultima_compra"].isna()

print(df["dias_desde_ultima_compra"].describe())
print("filas sin recencia:", df["dias_desde_ultima_compra_era_nulo"].sum())

última fecha válida: 2026-05-15 00:00:00
count    569878.000000
mean        262.496436
std          95.574646
min         157.000000
25%         189.000000
50%         234.000000
75%         308.000000
max         656.000000
Name: dias_desde_ultima_compra, dtype: float64
filas sin recencia: 1098


`fecha_ultima_compra` no existe en el dataset, por lo que se derivó como la fecha de pedido más reciente por `id_cliente` (≤ 2026-10-19). Se detectaron 14.694 pedidos con fecha en 2027 (posteriores a hoy), imposibles y excluidos del cálculo, y 20.288 fechas nulas. 993 clientes quedan sin recencia y se marcan con `dias_desde_ultima_compra_era_nulo` en lugar de imputar.

 codigo de verificacion , se uso para verificar la feature , no se usara en el notebook final 

In [8]:
def verificar_recencia(df, hoy, n_clientes=3, n_sin_recencia=2, semilla=42):
    """Revisa filas de ejemplo y recalcula dias_desde_ultima_compra de forma independiente.
    Solo diagnóstico: no modifica df."""
    cols = ["id_cliente", "fecha_pedido", "fecha_ultima_compra",
            "dias_desde_ultima_compra", "dias_desde_ultima_compra_era_nulo"]

    # 1) Clientes con recencia: todos sus pedidos ordenados
    con = df.loc[~df["dias_desde_ultima_compra_era_nulo"], "id_cliente"].drop_duplicates()
    for c in con.sample(n_clientes, random_state=semilla):
        print(df.loc[df["id_cliente"] == c, cols].sort_values("fecha_pedido"), "\n")

    # 2) Clientes sin recencia
    sin = df.loc[df["dias_desde_ultima_compra_era_nulo"], "id_cliente"].drop_duplicates().head(n_sin_recencia)
    for c in sin:
        print(df.loc[df["id_cliente"] == c, cols + ["fecha_pedido_futura"]].sort_values("fecha_pedido"), "\n")

    # 3) Recálculo independiente y comparación
    maximo = df.loc[df["fecha_pedido"] <= hoy].groupby("id_cliente")["fecha_pedido"].max()
    esperado = (hoy - df["id_cliente"].map(maximo)).dt.days
    coincide = esperado.fillna(-1).eq(df["dias_desde_ultima_compra"].fillna(-1)).all()
    print("coincide con el cálculo manual:", coincide)
    return coincide

### Feature dañada 2. cliente recurrente sin variable numero de compras

`numero_compras` no existe en el dataset, por lo que se derivó como el número de `id_pedido` distintos por `id_cliente` (todos los pedidos, incluidos los de fecha nula o futura, ya que la fecha no afecta la existencia de la compra). Se verificó que `id_pedido` es único (570.976 valores, sin nulos ni valores concatenados). `es_cliente_recurrente` marca clientes con más de una compra: ≈90,5 % de los clientes (≈97,5 % de las filas, porcentaje inflado porque los clientes recurrentes aportan más filas).

In [9]:
#Cliente recurrente
#df["es_cliente_recurrente"] = (df["numero_compras"] > 1)

In [10]:
compras = df.groupby("id_cliente")["id_pedido"].nunique().rename("numero_compras")

n0 = len(df)
df = df.merge(compras, on="id_cliente", how="left", validate="m:1")
assert len(df) == n0

df["es_cliente_recurrente"] = df["numero_compras"] > 1

print(df["numero_compras"].describe())
print(df["es_cliente_recurrente"].value_counts(normalize=True))

count    570976.000000
mean          4.684908
std           1.922838
min           1.000000
25%           3.000000
50%           5.000000
75%           6.000000
max          14.000000
Name: numero_compras, dtype: float64
es_cliente_recurrente
True     0.974861
False    0.025139
Name: proportion, dtype: float64


In [11]:
por_cliente = df.drop_duplicates("id_cliente")
print(len(por_cliente))
print(por_cliente["es_cliente_recurrente"].value_counts())
print(por_cliente["es_cliente_recurrente"].value_counts(normalize=True))

151117
es_cliente_recurrente
True     136763
False     14354
Name: count, dtype: int64
es_cliente_recurrente
True     0.905014
False    0.094986
Name: proportion, dtype: float64


### Feature 3. resumen cliente 

In [12]:
# Resumen cliente
"""
resumen_cliente = (
    df.groupby("id_cliente")
      .agg(
          total_gastado=("monto", "sum"),
          numero_compras=("id_compra", "count"),
          gasto_promedio=("monto", "mean"),
      )
      .reset_index()
)
"""

'\nresumen_cliente = (\n    df.groupby("id_cliente")\n      .agg(\n          total_gastado=("monto", "sum"),\n          numero_compras=("id_compra", "count"),\n          gasto_promedio=("monto", "mean"),\n      )\n      .reset_index()\n)\n'

In [13]:
resumen_cliente = (
    df.groupby("id_cliente")
      .agg(
          total_gastado=("monto_compra", "sum"),
          numero_compras=("id_pedido", "count"),
          gasto_promedio=("monto_compra", "mean"),
      )
      .reset_index()
)

 Feature: resumen_cliente

El dataset tiene una fila por pedido. Esta feature lo agrega a **una fila por cliente** (`id_cliente`), con tres variables que resumen su comportamiento de compra:

- `total_gastado`: suma de `monto_compra` de todos sus pedidos.
- `numero_compras`: cantidad de pedidos (`id_pedido`) del cliente.
- `gasto_promedio`: promedio de `monto_compra` por pedido.

La consigna usaba `monto` e `id_compra`, que no existen en este dataset; se reemplazaron por `monto_compra` (monto en unidades originales, no las versiones escaladas ni logarítmicas, que no tienen sentido al sumar) e `id_pedido` (único por fila, verificado). Se agrupa por `id_cliente` y no por correo porque este tiene faltantes. `sum` y `mean` ignoran los nulos de `monto_compra`. El resultado es una tabla aparte, `resumen_cliente`, con una fila por cliente.

In [14]:
import numpy as np

def verificar_resumen_cliente(df, resumen_cliente, n=3, semilla=42):
    """Diagnóstico de resumen_cliente. Solo lectura; no modifica df ni el resumen."""
    checks = {
        "una fila por cliente": len(resumen_cliente) == df["id_cliente"].nunique(),
        "id_cliente sin duplicados": resumen_cliente["id_cliente"].is_unique,
        "suma de compras = filas de df": resumen_cliente["numero_compras"].sum() == len(df),
        "total_gastado suma igual que df": np.isclose(
            resumen_cliente["total_gastado"].sum(), df["monto_compra"].sum()),
    }
    for nombre, ok in checks.items():
        print(f"{'OK ' if ok else 'FALLA'} | {nombre}")

    # Muestra de clientes: pedidos originales vs. valores del resumen
    for c in resumen_cliente["id_cliente"].sample(n, random_state=semilla):
        print("\n", df.loc[df["id_cliente"] == c, ["id_cliente", "id_pedido", "monto_compra"]])
        print(resumen_cliente.loc[resumen_cliente["id_cliente"] == c])

    return all(checks.values())

### feature de dominio 4 arreglada 

In [15]:
# RFM

# Se asume que resumen_cliente ya tiene: dias_desde_ultima_compra,
# numero_compras y total_gastado (de las lecciones anteriores).
"""
resumen_cliente["score_R"] = pd.qcut(
    resumen_cliente["dias_desde_ultima_compra"], q=4, labels=[4, 3, 2, 1]
)
resumen_cliente["score_F"] = pd.qcut(
    resumen_cliente["numero_compras"].rank(method="first"), q=4, labels=[1, 2, 3, 4]
)
resumen_cliente["score_M"] = pd.qcut(
    resumen_cliente["total_gastado"], q=4, labels=[1, 2, 3, 4]
)
resumen_cliente["rfm_score"] = (
    resumen_cliente["score_R"].astype(int)
    + resumen_cliente["score_F"].astype(int)
    + resumen_cliente["score_M"].astype(int)
)
 
"""

'\nresumen_cliente["score_R"] = pd.qcut(\n    resumen_cliente["dias_desde_ultima_compra"], q=4, labels=[4, 3, 2, 1]\n)\nresumen_cliente["score_F"] = pd.qcut(\n    resumen_cliente["numero_compras"].rank(method="first"), q=4, labels=[1, 2, 3, 4]\n)\nresumen_cliente["score_M"] = pd.qcut(\n    resumen_cliente["total_gastado"], q=4, labels=[1, 2, 3, 4]\n)\nresumen_cliente["rfm_score"] = (\n    resumen_cliente["score_R"].astype(int)\n    + resumen_cliente["score_F"].astype(int)\n    + resumen_cliente["score_M"].astype(int)\n)\n\n'

In [16]:
recencia = df.drop_duplicates("id_cliente")[["id_cliente", "dias_desde_ultima_compra"]]

resumen_cliente = resumen_cliente.merge(recencia, on="id_cliente", how="left", validate="1:1")
print(resumen_cliente[["dias_desde_ultima_compra", "numero_compras", "total_gastado"]].isna().sum())

dias_desde_ultima_compra    993
numero_compras                0
total_gastado                 0
dtype: int64


In [17]:
resumen_cliente["score_R"] = pd.qcut(
    resumen_cliente["dias_desde_ultima_compra"].rank(method="first"), q=4, labels=[4, 3, 2, 1]
)
resumen_cliente["score_F"] = pd.qcut(
    resumen_cliente["numero_compras"].rank(method="first"), q=4, labels=[1, 2, 3, 4]
)
resumen_cliente["score_M"] = pd.qcut(
    resumen_cliente["total_gastado"].rank(method="first"), q=4, labels=[1, 2, 3, 4]
)

resumen_cliente["rfm_score"] = (
    resumen_cliente["score_R"].astype(float)
    + resumen_cliente["score_F"].astype(float)
    + resumen_cliente["score_M"].astype(float)
).astype("Int64")

print(resumen_cliente["rfm_score"].describe())
print(resumen_cliente["rfm_score"].isna().sum())

count    150124.0
mean     7.518085
std      2.627798
min           3.0
25%           5.0
50%           8.0
75%          10.0
max          12.0
Name: rfm_score, dtype: Float64
993


In [18]:
for col in ["score_R", "score_F", "score_M"]:
    print(f"\n{col}")
    print(resumen_cliente[col].value_counts(dropna=False).sort_index())

print("\nrfm_score")
print(resumen_cliente["rfm_score"].value_counts(dropna=False).sort_index())

# Muestra de clientes con sus valores originales y sus puntajes
print(resumen_cliente[["id_cliente", "dias_desde_ultima_compra", "score_R",
                       "numero_compras", "score_F",
                       "total_gastado", "score_M", "rfm_score"]].sample(8, random_state=42))


score_R
score_R
4      37531
3      37531
2      37531
1      37531
NaN      993
Name: count, dtype: int64

score_F
score_F
1    37780
2    37779
3    37779
4    37779
Name: count, dtype: int64

score_M
score_M
1    37780
2    37779
3    37779
4    37779
Name: count, dtype: int64

rfm_score
rfm_score
3       12133
4       12182
5       14349
6       16867
7       17580
8       18144
9       18527
10      17252
11      13984
12       9106
<NA>      993
Name: count, dtype: Int64
       id_cliente  dias_desde_ultima_compra score_R  numero_compras score_F  \
6827      C106354                     205.0       3               4       2   
64503     C160176                     300.0       2               4       3   
56720     C152913                     251.0       3               6       4   
16540     C115397                     325.0       2               5       3   
140271     C88866                     227.0       3               3       2   
54145     C150525                     267.0

 Feature: segmentación RFM (rfm_score)

**Qué es.** RFM es una técnica clásica de marketing para puntuar a cada cliente según tres dimensiones de su comportamiento de compra:

- **R (Recency, recencia):** hace cuánto compró por última vez (`dias_desde_ultima_compra`). Menos días es mejor.
- **F (Frequency, frecuencia):** cuántas veces compró (`numero_compras`). Más compras es mejor.
- **M (Monetary, valor monetario):** cuánto gastó en total (`total_gastado`). Más gasto es mejor.

**Qué hace el código.** Trabaja sobre `resumen_cliente`, que tiene una fila por cliente (151.117 filas), y sigue estos pasos:

1. **Traer la recencia al resumen.** `dias_desde_ultima_compra` se calculó a nivel pedido en `df`, así que se toma una fila por cliente (`drop_duplicates("id_cliente")`) y se une al resumen con `merge` (`validate="1:1"`).
2. **Dividir cada variable en cuartiles.** Con `pd.qcut(..., q=4)` se reparte a los clientes en 4 grupos de tamaño casi igual. Formalmente, para una variable $x$ con cuartiles $Q_1 < Q_2 < Q_3$, el puntaje de un cliente es:

$$
\text{score}(x) = k \quad \text{si } x \text{ cae en el } k\text{-ésimo cuartil}, \quad k \in \{1,2,3,4\}
$$

3. **Orientar los puntajes para que 4 sea siempre "mejor".** En F y M, los valores altos reciben 4 (etiquetas `[1, 2, 3, 4]`). En R la escala se **invierte** (etiquetas `[4, 3, 2, 1]`), porque un cliente con pocos días desde su última compra es más reciente y debe puntuar más alto.
4. **Desempatar con `rank(method="first")`.** La recencia y `numero_compras` toman pocos valores distintos (de 1 a 14 compras, por ejemplo), y `qcut` falla con `Bin edges must be unique` cuando hay muchos empates en los bordes. `rank(method="first")` convierte los valores en un ranking sin empates, asignando el orden según la posición de aparición, y así `qcut` puede formar los 4 grupos.
5. **Sumar los tres puntajes.**

$$
\texttt{rfm\_score} = \text{score}_R + \text{score}_F + \text{score}_M, \qquad \texttt{rfm\_score} \in \{3, 4, \dots, 12\}
$$

Un cliente con 12 es reciente, frecuente y de alto gasto; uno con 3 es lo opuesto. Se convierte a `Int64` (entero que admite nulos) porque algunos clientes no tienen puntaje.

**Decisiones y limitaciones.**

- **Clientes sin recencia (993).** No tienen ninguna fecha de pedido válida (todas nulas o de 2027, imposibles), por lo que `score_R` y `rfm_score` quedan en `NaN`. No se imputa un puntaje, porque inventarlo ocultaría la falta de información.
- **Desempate arbitrario.** Con `rank(method="first")`, dos clientes con el mismo valor pueden quedar en cuartiles distintos solo por su orden en la tabla. Es una limitación de la técnica, más visible en `numero_compras`, que tiene pocos valores distintos.
- **Desfase de fechas.** Los datos válidos terminan el 2026-05-15 y el corte de referencia es el 2026-10-19, así que la recencia mínima es de 157 días. R mantiene el orden relativo entre clientes, pero no indica abandono real.
- **Posible fuga de información.** R, F y M usan todo el historial del cliente. Si se usan para predecir `fuga_cliente` con la partición `conjunto`, habría que calcularlas solo con datos de entrenamiento o con una ventana previa a cada pedido.
- **Resultado.** Es una variable de dominio que resume el valor del cliente en un solo número de 3 a 12, y que permite segmentar (por ejemplo, clientes de alto valor frente a clientes en riesgo).

### tras features de dominio típicas del sector retail
Estacionalidad: bandera es_temporada_alta (por ejemplo, noviembre y diciembre en NovaMarket).
Afinidad de categoría: categoria_favorita, la categoría de producto donde el cliente concentra más compras.
Indicador de canal preferido: canal_principal (web, app o tienda física), derivado de contar en qué canal ocurren más compras.

In [19]:
"""
# Es temporada alta
df_transacciones["fecha_pedido"] = pd.to_datetime(df_transacciones["fecha"])
# Bandera a nivel de transacción indicando si el mes es Noviembre (11) o Diciembre (12)
df_transacciones["es_temporada_alta"] = df_transacciones["fecha_pedido"].dt.month.isin([11, 12]).astype(int)

# Categoría favorita
cat_conteo = (
    df_transacciones.groupby(["id_cliente", "categoria_producto"])
    .size()
    .reset_index(name="conteo_compras")
)

# Ordenar de mayor a menor y quedarse con la primera de cada cliente (la favorita)
categoria_favorita = (
    cat_conteo.sort_values(["id_cliente", "conteo_compras"], ascending=[True, False])
    .drop_duplicates(subset=["id_cliente"])
    [["id_cliente", "categoria_producto"]]
    .rename(columns={"categoria": "categoria_favorita"})
)

# Canal principal
# Contar transacciones por cliente y canal
canal_conteo = (
    df_transacciones.groupby(["id_cliente", "canal_compra"])
    .size()
    .reset_index(name="conteo_canal")
)

# Seleccionar el canal con mayor frecuencia por cliente
canal_principal = (
    canal_conteo.sort_values(["id_cliente", "conteo_canal"], ascending=[True, False])
    .drop_duplicates(subset=["id_cliente"])
    [["id_cliente", "canal_compra"]]
    .rename(columns={"canal": "canal_principal"})
)
"""

'\n# Es temporada alta\ndf_transacciones["fecha_pedido"] = pd.to_datetime(df_transacciones["fecha"])\n# Bandera a nivel de transacción indicando si el mes es Noviembre (11) o Diciembre (12)\ndf_transacciones["es_temporada_alta"] = df_transacciones["fecha_pedido"].dt.month.isin([11, 12]).astype(int)\n\n# Categoría favorita\ncat_conteo = (\n    df_transacciones.groupby(["id_cliente", "categoria_producto"])\n    .size()\n    .reset_index(name="conteo_compras")\n)\n\n# Ordenar de mayor a menor y quedarse con la primera de cada cliente (la favorita)\ncategoria_favorita = (\n    cat_conteo.sort_values(["id_cliente", "conteo_compras"], ascending=[True, False])\n    .drop_duplicates(subset=["id_cliente"])\n    [["id_cliente", "categoria_producto"]]\n    .rename(columns={"categoria": "categoria_favorita"})\n)\n\n# Canal principal\n# Contar transacciones por cliente y canal\ncanal_conteo = (\n    df_transacciones.groupby(["id_cliente", "canal_compra"])\n    .size()\n    .reset_index(name="conteo

#### -Es temporada alta 

In [20]:
valida = df["fecha_pedido"].notna() & ~df["fecha_pedido_futura"]

df["es_temporada_alta"] = (
    df["fecha_pedido"].dt.month.isin([11, 12]).astype(float).where(valida).astype("Int64")
)
print(df["es_temporada_alta"].value_counts(dropna=False))

es_temporada_alta
0       470363
1        65631
<NA>     34982
Name: count, dtype: Int64


 Feature: es_temporada_alta

Bandera a nivel de pedido: vale 1 si `fecha_pedido` cae en noviembre o diciembre (periodo de mayor consumo por campañas y fiestas) y 0 en cualquier otro mes.

La consigna usaba `df_transacciones` y `fecha`, que no existen; se utilizó `fecha_pedido`, que ya es datetime. Las filas sin fecha válida (20.288 nulas y 14.694 con fecha en 2027, imposible) quedan en `NA` y no en 0, porque marcarlas como "no es temporada alta" afirmaría algo falso. Resultado: 65.631 pedidos en temporada alta (≈12,2 % de los pedidos con fecha válida), 470.363 fuera y 34.982 sin dato.

Limitación: las fechas válidas terminan en mayo de 2026, por lo que solo noviembre y diciembre de 2025 están representados.

#### - categoria favorita

In [21]:
cat_conteo = (
    df.groupby(["id_cliente", "categoria_producto"])
      .size()
      .reset_index(name="conteo_compras")
)

categoria_favorita = (
    cat_conteo.sort_values(["id_cliente", "conteo_compras", "categoria_producto"],
                           ascending=[True, False, True])
    .drop_duplicates(subset="id_cliente")
    [["id_cliente", "categoria_producto"]]
    .rename(columns={"categoria_producto": "categoria_favorita"})
)

# Cuántos clientes tienen empate en la categoría más comprada
maximo = cat_conteo.groupby("id_cliente")["conteo_compras"].transform("max")
n_top = (cat_conteo["conteo_compras"] == maximo).groupby(cat_conteo["id_cliente"]).sum()
print("clientes con empate:", (n_top > 1).sum(), f"({(n_top > 1).mean():.1%})")

clientes con empate: 68175 (45.1%)


#### -canal principal

In [22]:
cols_canal = [c for c in df.columns if c.startswith("canal_")]
print(cols_canal)
print(df[cols_canal].sum(axis=1).value_counts())   # debe ser 1 en todas las filas

activa = df[cols_canal].astype(bool)
df["canal_compra"] = (
    activa.idxmax(axis=1).str.replace("canal_", "", regex=False)
    .where(activa.any(axis=1))
    .str.strip().str.lower()
)

canal_conteo = (
    df.groupby(["id_cliente", "canal_compra"])
      .size()
      .reset_index(name="conteo_canal")
)

canal_principal = (
    canal_conteo.sort_values(["id_cliente", "conteo_canal", "canal_compra"],
                             ascending=[True, False, True])
    .drop_duplicates(subset="id_cliente")
    [["id_cliente", "canal_compra"]]
    .rename(columns={"canal_compra": "canal_principal"})
)

['canal_App', 'canal_Marketplace', 'canal_Tienda', 'canal_Web']
1    570976
Name: count, dtype: int64


se van para resumen cliente 

Features: categoria_favorita y canal_principal

Variables por cliente que resumen sus preferencias de compra:

- `categoria_favorita`: la categoría con más pedidos del cliente.
- `canal_principal`: el canal con más pedidos del cliente.

Para cada cliente $i$ y cada valor posible $k$ (categoría o canal), con $n_{ik}$ el número de pedidos del cliente en $k$:

$$
\texttt{favorita}_i = \arg\max_k \; n_{ik}
$$

**Qué hace el código.** Cuenta pedidos por (`id_cliente`, categoría) o (`id_cliente`, canal) con `groupby().size()`, ordena de mayor a menor y conserva la primera fila de cada cliente con `drop_duplicates`. La consigna usaba `canal_compra`, que no existe: el canal estaba en las dummies `canal_*`, por lo que se reconstruyó antes de contar (se verificó que cada pedido tiene exactamente una dummy activa). Además, los `rename` de la consigna apuntaban a nombres inexistentes y no tenían efecto; se corrigieron.

**Empates.** El 45,1 % de los clientes (68.175) tiene empate en la categoría más comprada, porque la mayoría compra pocas veces repartidas entre 6 categorías. Para que el resultado sea reproducible, el desempate es alfabético, lo que sesga hacia las primeras categorías sin reflejar una preferencia real. Por eso se agregan los indicadores `categoria_favorita_empate` y `canal_principal_empate` (canal: [completar con la salida]), para poder filtrar o ignorar los clientes sin favorita clara.

**Limitación.** Ambas variables usan todo el historial del cliente; si se emplean para predecir `fuga_cliente` con la partición `conjunto`, habría que calcularlas solo con datos de entrenamiento.

In [23]:
resumen_cliente = (
    resumen_cliente
    .merge(categoria_favorita, on="id_cliente", how="left", validate="1:1")
    .merge(canal_principal,    on="id_cliente", how="left", validate="1:1")
)
print(resumen_cliente.shape)

(151117, 11)


In [24]:
# Categoría (n_top ya calculado en el bloque anterior)
emp_cat = (n_top > 1).rename("categoria_favorita_empate")

# Canal
maximo_c = canal_conteo.groupby("id_cliente")["conteo_canal"].transform("max")
n_top_c = (canal_conteo["conteo_canal"] == maximo_c).groupby(canal_conteo["id_cliente"]).sum()
emp_canal = (n_top_c > 1).rename("canal_principal_empate")

resumen_cliente = (
    resumen_cliente
    .merge(emp_cat.reset_index(),   on="id_cliente", how="left", validate="1:1")
    .merge(emp_canal.reset_index(), on="id_cliente", how="left", validate="1:1")
)
print(resumen_cliente[["categoria_favorita_empate", "canal_principal_empate"]].mean())

categoria_favorita_empate    0.451141
canal_principal_empate       0.365856
dtype: float64


**Empates.** El 45,1 % de los clientes tiene empate en la categoría más comprada y el 36,6 % en el canal más usado, porque la mayoría compra pocas veces (≈3,8 pedidos en promedio) repartidas entre 6 categorías y 4 canales. Para que el resultado sea reproducible, el desempate es alfabético, lo que sesga hacia las primeras categorías o canales sin reflejar una preferencia real. Por eso se agregan los indicadores `categoria_favorita_empate` y `canal_principal_empate`, que permiten filtrar o ignorar los clientes sin favorito claro.

## Resumen general de features

Se construyeron **9 features** (más sus indicadores auxiliares) sobre el dataset integrado de ventas, bodega y CRM (570.976 pedidos, 151.117 clientes). Ninguna de las consignas se pudo ejecutar tal cual, porque usaban columnas que no existen en este dataset (`fecha_ultima_compra`, `numero_compras`, `monto`, `id_compra`, `fecha`, `canal_compra`, `df_transacciones`). En cada caso se derivó la variable desde columnas reales o se mapeó al nombre correcto.

### Features creadas

| Feature | Nivel | Dónde vive | Cómo se construyó |
|---|---|---|---|
| `dias_desde_ultima_compra` (+ `fecha_ultima_compra`, `dias_desde_ultima_compra_era_nulo`) | cliente | `df` y `resumen_cliente` | Máximo de `fecha_pedido` válida por `id_cliente`; días hasta el corte 2026-10-19. Los clientes sin fecha válida quedan en `NaN` con indicador. |
| `numero_compras` | cliente | `df` y `resumen_cliente` | Conteo de `id_pedido` distintos por `id_cliente` (1 a 14). |
| `es_cliente_recurrente` | cliente | `df` | `numero_compras > 1`. |
| `total_gastado`, `gasto_promedio` | cliente | `resumen_cliente` | Suma y promedio de `monto_compra` por `id_cliente`. |
| `score_R`, `score_F`, `score_M`, `rfm_score` | cliente | `resumen_cliente` | Cuartiles (`qcut` con `rank(method="first")`) de recencia (escala invertida), frecuencia y gasto; suma de 3 a 12. |
| `es_temporada_alta` | pedido | `df` | 1 si el mes de `fecha_pedido` es noviembre o diciembre; `NA` si la fecha es nula o futura. |
| `categoria_favorita` (+ `categoria_favorita_empate`) | cliente | `resumen_cliente` | Categoría con más pedidos por cliente, reconstruida antes desde las dummies `cat_*`. |
| `canal_principal` (+ `canal_principal_empate`) | cliente | `resumen_cliente` | Canal con más pedidos por cliente, reconstruido antes desde las dummies `canal_*`. |

**Feature descartada:** `compras_ultimos_30_dias`. La ventana de 30 días previa al corte (2026-09-19 a 2026-10-19) no contiene ningún pedido válido, porque los datos terminan el 2026-05-15. La feature sería constante en 0, o contaría solo fechas de 2027 (imposibles), por lo que no aporta información.

### Qué se usó para construirlas

- **Llave de agregación:** `id_cliente`, que no tiene nulos. No se usó `correo_cliente`, porque tiene faltantes (MNAR).
- **Mapeo de nombres:** `monto` → `monto_compra` (valor original, no las versiones escaladas, logarítmica o Box-Cox, que no tienen sentido al sumar); `id_compra` → `id_pedido`, que es único por fila (570.976 valores, sin nulos ni valores concatenados); `fecha` → `fecha_pedido`.
- **Reconstrucción desde dummies:** `categoria` y `canal_compra` se recuperaron con `idxmax` sobre las columnas `cat_*` y `canal_*`, verificando que cada fila tuviera exactamente una dummy activa.
- **Joins:** se hicieron con `how="left"` y `validate`, comprobando que el número de filas no cambiara.
- **Indicadores `*_era_nulo` y `*_empate`:** se conservó la trazabilidad en lugar de imputar valores inventados.

### Decisiones sobre la calidad de los datos

- **Fechas imposibles:** 14.694 pedidos con fecha en 2027 se trataron como inválidos (columna `fecha_pedido_futura`) y se excluyeron de la recencia y de `es_temporada_alta`. Además hay 20.288 fechas nulas.
- **Clientes sin recencia:** 993 clientes (1.098 filas) no tienen ninguna fecha válida; quedan sin `dias_desde_ultima_compra` ni `rfm_score`, sin imputar.
- **Empates:** el 45,1 % de los clientes tiene empate en la categoría favorita y el 36,6 % en el canal principal. El desempate es alfabético (para que sea reproducible) y se marca con indicadores.

### Limitaciones a tener en cuenta

1. **Desfase temporal:** las fechas válidas terminan el 2026-05-15 y el corte es el 2026-10-19, por lo que la recencia mínima es de 157 días. R conserva el orden relativo entre clientes, pero no indica abandono real. Del mismo modo, `es_temporada_alta` solo cubre noviembre y diciembre de 2025.
2. **Porcentajes por fila:** el 97,5 % de las filas pertenece a clientes recurrentes, pero ese número está inflado, porque los clientes recurrentes aportan más filas. La cifra por cliente es menor y es la que debe reportarse.
3. **Fuga de información:** las features por cliente (recencia, compras, gasto, RFM, categoría y canal favoritos) usan todo el historial, incluidos pedidos posteriores a cada fila. Si se usan para predecir `fuga_cliente` con la partición `conjunto`, deben recalcularse solo con datos de entrenamiento o con una ventana previa a cada pedido.
4. **Columnas contaminadas por fechas imposibles:** `mes_pedido`, `trimestre_pedido`, `dia_semana_pedido` y `es_fin_semana_pedido` se calcularon con las fechas de 2027 en esas 14.694 filas y no se corrigieron en este notebook.
5. **Desempate arbitrario:** `rank(method="first")` en RFM desempata por orden de aparición, lo que afecta sobre todo a `numero_compras`, que tiene pocos valores distintos.

Si exportamos tal cual, el CSV no incluirá la mayoría de las features ya que viven en resumen_cliente, y no en df. Por eso conviene unirlas antes de exportar.

In [25]:
import os

# 1) Columnas de resumen_cliente que todavía no están en df
#    (numero_compras y dias_desde_ultima_compra ya están, y unirlas las duplicaría)
cols_nuevas = [c for c in resumen_cliente.columns if c not in df.columns]

# 2) Unir a nivel pedido: cada cliente repite sus valores en todos sus pedidos
n0 = len(df)
df_final = df.merge(
    resumen_cliente[["id_cliente"] + cols_nuevas],
    on="id_cliente", how="left", validate="m:1",
)
assert len(df_final) == n0, "El merge cambió el número de filas"

# 3) Exportar
os.makedirs("Datos", exist_ok=True)
df_final.to_csv("Datos/novamarket_con_features.csv", index=False)

# 4) Verificación: recargar y comparar dimensiones
df_check = pd.read_csv("Datos/novamarket_con_features.csv")
print(df_final.shape, df_check.shape, df_final.shape == df_check.shape)
print(cols_nuevas)

(570976, 84) (570976, 84) True
['total_gastado', 'gasto_promedio', 'score_R', 'score_F', 'score_M', 'rfm_score', 'categoria_favorita', 'canal_principal', 'categoria_favorita_empate', 'canal_principal_empate']
